# N137 · 懒传播、动态节点与更新复合

**目标：** 正确组合懒标记并维护区间统计量。

**先修：** N136, N034, N124。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** range add/flip；push/pull；标记复合次序；区间长度；动态开点；赋值扩展。

**配套讲解来源：** [同名逐章意见](../../comment/137_lazy_segment_tree_range_updates.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

上一章的线段树只能"改一个点"，这一章升级成"整段一起改"：给区间 `[l, r)` 里每个数都加 delta，或者把一段 01 串整体翻转（0 变 1、1 变 0），然后随时查任意一段的和或 1 的个数。如果区间加真去逐个叶子改，一次操作最坏 O(n)。懒传播（lazy propagation）的思路是：如果一个节点管辖的整段都被覆盖，就别往下走了——节点自己的和可以直接算出来（增量 × 区间长度），把"孩子们还欠着的更新"记在一个 lazy 标记里，等哪天真的需要往下走了再补。

一个具体到数字的例子（就是 “运行示例”部分 的）：数组 `[1, 2, 3, 4]` 建树。执行 `add(0, 4, 2)`（整段加 2），逐点值变成 `[3, 4, 5, 6]`——整段覆盖，根节点直接加 `4 × 2 = 8`，一次搞定。再执行 `add(1, 3, -1)`，变成 `[3, 3, 4, 6]`。再 `add(2, 4, 5)`，变成 `[3, 3, 9, 11]`。每一步操作都只碰 O(log n) 个节点，但查询任何位置的值都是对的。

## 2. 基础知识：先读懂这些词

- **懒传播（lazy propagation）**：把"还没下传给孩子"的更新存在节点的 lazy 标记里，能不动孩子就不动。你需要它是因为区间操作如果真的一路改到叶子，代价和暴力没区别；挂起标记后，整段覆盖的操作停在半路就完成。
- **range add / range flip**：区间加（整段每个数加 delta）和区间翻转（整段 0/1 互换）是两种最经典的区间更新，本章各实现一个版本。
- **区间长度参与计算**：整段加 delta 时，节点和的变化量是 `delta × (r - l)`，因为节点管着 `r - l` 个元素。这就是 `_apply` 里 `(r-l)*delta` 的来历。
- **push / pull**：push 是"下传标记"——访问孩子之前，把孩子欠的更新先补给他们；pull 是"上拉重算"——孩子变了之后，父亲用两个孩子的新值重新算自己。口诀：下去前 push，上来后 pull。
- **标记复合次序**：多个待下传的标记要能合并成一个。加法标记直接相加（加 3 再加 5 等于加 8）；翻转标记按异或复合（翻两次等于没翻，翻三次等于翻一次）。注意：赋值和加法不可交换（先赋 5 再加 2 得 7，先加 2 再赋 5 得 5），所以赋值扩展不能直接套本章的相加标记。
- **翻转下的计数**：翻转一段 0/1 后，该段 1 的个数变成"长度减去原来的 1 的个数"，比如长度 4、有 1 个 1，翻转后有 3 个 1。
- **动态开点（dynamic node creation）**：当坐标范围大到 10^9 但实际操作很少时，不开 4n 的数组，而是"用到哪个节点才创建哪个"。本章代码是静态版，动态开点属于拓展方向（notebook 明说不宣称已实现）。
- **闭区间转半开**：外部题目给 `[a, b]` 闭区间时，树内部统一用 `[a, b+1)` 半开区间处理。

## 3. 思路推导：从小例子开始

- **Step 1（建树）**：用递归把 `[0, n)` 一分为二建树，每个节点存自己区间的和。开 `4n` 大的数组是因为递归二分的满形态需要这个上界。
- **Step 2（_apply：单节点打标记）**：对一个节点执行"整段加 delta"，只做两件事：`total[node] += (r-l)*delta`（自己和立刻对，孩子先欠着）以及 `lazy[node] += delta`（记账）。
- **Step 3（_push：下传）**：要去访问孩子了，先看节点有没有欠账：`lazy[node]` 非零且不是叶子，就把标记 _apply 到两个孩子身上，然后清零自己的标记。账就这么一层层往下传。
- **Step 4（add：区间更新）**：递归三种情况——当前节点区间和目标区间完全不相交（直接返回）；完全被覆盖（_apply 打标记就收工）；部分相交（先 _push 下传，再递归左右孩子，回来后 pull：`total[node] = 左 + 右`）。
- **Step 5（query：区间查询）**：和 add 同构的三分递归；不相交返回 0，整段覆盖直接返回 `total[node]`（不需要 push！因为 total 已经包含了自身所有待下传更新的效果），部分相交则 push 后递归求和。
- **Step 6（FlipCountTree：翻转版）**：继承加法树，只重写 `_apply`：delta 为奇数时 `total = (r-l) - total`（1 的个数翻转），`lazy ^= 1`（翻转标记按异或复合，两次抵消）。`flip` 就是 `add(l, r, 1)`，`count` 就是 `query`。

**手算演示（“运行示例”部分 的表）**：数组 `[1,2,3,4]`（size 4，根管 [0,4)，两个孩子分别管 [0,2) 和 [2,4)）。

| 操作 | 内部发生了什么 | 实际逐点值 |
|---|---|---|
| 初始 | 建树，根 total=10 | [1, 2, 3, 4] |
| [0,4) 加 2 | 整段覆盖根节点：total = 10 + 4×2 = 18，根打 lazy=2，孩子先欠着 | [3, 4, 5, 6] |
| [1,3) 加 -1 | 与两个孩子都部分相交：先把根的 lazy=2 push 给两个孩子（各自 total 加 2×2），再各自递归、下到叶子附近收尾，最后逐层 pull | [3, 3, 4, 6] |
| [2,4) 加 5 | 恰好整段覆盖右孩子 [2,4)：total += 2×5，打标记 | [3, 3, 9, 11] |

注意第一行操作后我们从未真正改过叶子，但后面查单个位置的值（`query(i, i+1)`）时 push 机制会沿途把欠账补齐，所以查出来的值永远正确。

## 4. 核心代码：RangeAddSumTree

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
class RangeAddSumTree:

    def __init__(self, values):
        values = list(values)
        self.n = len(values)
        self.total = [0] * (4 * max(1, self.n))
        self.lazy = [0] * len(self.total)

        def build(node, l, r):
            if r - l == 1:
                self.total[node] = values[l]
                return
            mid = (l + r) // 2
            build(node * 2, l, mid)
            build(node * 2 + 1, mid, r)
            self.total[node] = self.total[node * 2] + self.total[node * 2 + 1]
        if self.n:
            build(1, 0, self.n)

    def _apply(self, node, l, r, delta):
        self.total[node] += (r - l) * delta
        self.lazy[node] += delta

    def _push(self, node, l, r):
        if self.lazy[node] and r - l > 1:
            mid = (l + r) // 2
            tag = self.lazy[node]
            self._apply(node * 2, l, mid, tag)
            self._apply(node * 2 + 1, mid, r, tag)
            self.lazy[node] = 0

    def add(self, left, right, delta):
        if not 0 <= left <= right <= self.n:
            raise IndexError('invalid half-open range')
        if left == right:
            return

        def update(node, l, r):
            if right <= l or r <= left:
                return
            if left <= l and r <= right:
                self._apply(node, l, r, delta)
                return
            self._push(node, l, r)
            mid = (l + r) // 2
            update(node * 2, l, mid)
            update(node * 2 + 1, mid, r)
            self.total[node] = self.total[node * 2] + self.total[node * 2 + 1]
        update(1, 0, self.n)

    def query(self, left, right):
        if not 0 <= left <= right <= self.n:
            raise IndexError('invalid half-open range')
        if left == right:
            return 0

        def get(node, l, r):
            if right <= l or r <= left:
                return 0
            if left <= l and r <= right:
                return self.total[node]
            self._push(node, l, r)
            mid = (l + r) // 2
            return get(node * 2, l, mid) + get(node * 2 + 1, mid, r)
        return get(1, 0, self.n)
```

### 逐段读懂代码

“分段实现”部分 有三个部分：RangeAddSumTree 类、FlipCountTree 子类、handle_sum_queries 函数。

**构造函数与建树**

```python
def __init__(self,values):
        values=list(values); self.n=len(values); self.total=[0]*(4*max(1,self.n)); self.lazy=[0]*len(self.total)
        def build(node,l,r):
            if r-l==1: self.total[node]=values[l]; return
            mid=(l+r)//2; build(node*2,l,mid); build(node*2+1,mid,r); self.total[node]=self.total[node*2]+self.total[node*2+1]
        if self.n: build(1,0,self.n)
```

先拷贝输入（不共享引用），`4*max(1,self.n)` 给足递归二分所需的数组上界（`max(1,·)` 兜住 n=0，空数组也合法）。内部函数 `build(node,l,r)` 管半开区间 `[l, r)`：`r-l==1` 说明只剩一个元素（叶子），存值返回；否则取中点递归建左右孩子，再用孩子之和填自己。`if self.n:` 保证空输入直接跳过建树（后面 query(0,0) 也成立）。

**打标记 `_apply` 与下传 `_push`**

```python
def _apply(self,node,l,r,delta): self.total[node]+=(r-l)*delta; self.lazy[node]+=delta
    def _push(self,node,l,r):
        if self.lazy[node] and r-l>1:
            mid=(l+r)//2; tag=self.lazy[node]
            self._apply(node*2,l,mid,tag); self._apply(node*2+1,mid,r,tag); self.lazy[node]=0
```

`_apply` 是"对一个节点整段加 delta"：自己的和立刻按元素个数放大（`(r-l)*delta`），同时把 delta 记到 lazy 上（孩子还欠着这笔账）。`_push` 的条件 `self.lazy[node] and r-l>1` 说的是：有欠账且不是叶子才需要下传；下传就是把同一个 tag 分别 _apply 给两个孩子（加法标记可以原样分下去、也可以先相加再分），最后清零自己的标记，避免重复发账。

**区间加 `add`**

```python
def add(self,left,right,delta):
        if not 0<=left<=right<=self.n: raise IndexError('invalid half-open range')
        if left==right: return
        def update(node,l,r):
            if right<=l or r<=left: return
            if left<=l and r<=right: self._apply(node,l,r,delta); return
            self._push(node,l,r); mid=(l+r)//2; update(node*2,l,mid); update(node*2+1,mid,r)
            self.total[node]=self.total[node*2]+self.total[node*2+1]
        update(1,0,self.n)
```

第一行挡非法半开区间；`if left==right: return` 处理空区间（什么都不加）。内部递归 `update(node,l,r)` 的三分支顺序是关键：完全不相交就返回；`left<=l and r<=right` 表示当前节点整段被目标覆盖，`_apply` 打完标记就收工（这就是懒传播省时间的地方）；否则"部分相交"，必须先 `_push` 把旧欠账发给孩子（否则新 delta 和旧标记会在孩子那里错乱），递归两个孩子，最后一句 pull 重算自己。

**区间查询 `query`**

```python
def query(self,left,right):
        if not 0<=left<=right<=self.n: raise IndexError('invalid half-open range')
        if left==right: return 0
        def get(node,l,r):
            if right<=l or r<=left: return 0
            if left<=l and r<=right: return self.total[node]
            self._push(node,l,r); mid=(l+r)//2
            return get(node*2,l,mid)+get(node*2+1,mid,r)
        return get(1,0,self.n)
```

前两行照旧挡非法区间和空区间（空区间和为 0）。内部 `get(node,l,r)` 和 update 同构：不相交返回 0；整段被覆盖直接 `return self.total[node]`——这里不用 push，因为 total 本来就包含了自己 lazy 的效果；部分相交才 push 后分治求和。

**FlipCountTree 子类**

```python
class FlipCountTree(RangeAddSumTree):

    def __init__(self, values):
        values = list(values)
        if any((x not in (0, 1) for x in values)):
            raise ValueError('binary values required')
        super().__init__(values)

    def _apply(self, node, l, r, delta):
        if delta & 1:
            self.total[node] = r - l - self.total[node]
            self.lazy[node] ^= 1

    def flip(self, left, right):
        self.add(left, right, 1)

    def count(self, left, right):
        return self.query(left, right)
```

构造时先挡非 0/1 的输入（`raise ValueError('binary values required')`），再调父类建树。妙处在于只重写了 `_apply` 这一个"打标记"原语：`delta&1` 只看奇偶（翻偶数次等于没翻），是奇数就把"1 的个数"换成"长度减 1 的个数"，并把翻转标记异或上 1（`^=1` 实现两次翻转抵消）。`flip` 和 `count` 只是给继承来的 add/query 换了名字——整套递归、push、pull 逻辑原封不动复用，这就是面向"更新原语"编程的好处。

**题目函数 `handle_sum_queries`**

```python
def handle_sum_queries(nums1, nums2, queries):
    tree = FlipCountTree(nums1)
    total = sum(nums2)
    out = []
    for kind, a, b in queries:
        if kind == 1:
            tree.flip(a, b + 1)
        elif kind == 2:
            total += a * tree.total[1]
        elif kind == 3:
            out.append(total)
        else:
            raise ValueError('unknown query type')
    return out
```

这是 LeetCode 2569 的直接实现：nums1 建翻转树，nums2 只需维护一个总和 total。类型 1 把闭区间 `[a,b]` 翻转——注意 `tree.flip(a,b+1)`，闭区间转半开要加一；类型 2 执行 `total += a × (nums1 中 1 的个数)`，而 1 的个数就是根节点的 `tree.total[1]`，O(1) 读取；类型 3 把当前 total 记入输出。未知类型抛 ValueError 防呆。

## 5. 分段实现：按顺序运行

**本章接口：** `RangeAddSumTree`、`FlipCountTree`、`handle_sum_queries(nums1, nums2, queries)`

### RangeAddSumTree

In [1]:
class RangeAddSumTree:

    def __init__(self, values):
        values = list(values)
        self.n = len(values)
        self.total = [0] * (4 * max(1, self.n))
        self.lazy = [0] * len(self.total)

        def build(node, l, r):
            if r - l == 1:
                self.total[node] = values[l]
                return
            mid = (l + r) // 2
            build(node * 2, l, mid)
            build(node * 2 + 1, mid, r)
            self.total[node] = self.total[node * 2] + self.total[node * 2 + 1]
        if self.n:
            build(1, 0, self.n)

    def _apply(self, node, l, r, delta):
        self.total[node] += (r - l) * delta
        self.lazy[node] += delta

    def _push(self, node, l, r):
        if self.lazy[node] and r - l > 1:
            mid = (l + r) // 2
            tag = self.lazy[node]
            self._apply(node * 2, l, mid, tag)
            self._apply(node * 2 + 1, mid, r, tag)
            self.lazy[node] = 0

    def add(self, left, right, delta):
        if not 0 <= left <= right <= self.n:
            raise IndexError('invalid half-open range')
        if left == right:
            return

        def update(node, l, r):
            if right <= l or r <= left:
                return
            if left <= l and r <= right:
                self._apply(node, l, r, delta)
                return
            self._push(node, l, r)
            mid = (l + r) // 2
            update(node * 2, l, mid)
            update(node * 2 + 1, mid, r)
            self.total[node] = self.total[node * 2] + self.total[node * 2 + 1]
        update(1, 0, self.n)

    def query(self, left, right):
        if not 0 <= left <= right <= self.n:
            raise IndexError('invalid half-open range')
        if left == right:
            return 0

        def get(node, l, r):
            if right <= l or r <= left:
                return 0
            if left <= l and r <= right:
                return self.total[node]
            self._push(node, l, r)
            mid = (l + r) // 2
            return get(node * 2, l, mid) + get(node * 2 + 1, mid, r)
        return get(1, 0, self.n)

### FlipCountTree

In [2]:
class FlipCountTree(RangeAddSumTree):

    def __init__(self, values):
        values = list(values)
        if any((x not in (0, 1) for x in values)):
            raise ValueError('binary values required')
        super().__init__(values)

    def _apply(self, node, l, r, delta):
        if delta & 1:
            self.total[node] = r - l - self.total[node]
            self.lazy[node] ^= 1

    def flip(self, left, right):
        self.add(left, right, 1)

    def count(self, left, right):
        return self.query(left, right)

### handle_sum_queries

In [3]:
def handle_sum_queries(nums1, nums2, queries):
    tree = FlipCountTree(nums1)
    total = sum(nums2)
    out = []
    for kind, a, b in queries:
        if kind == 1:
            tree.flip(a, b + 1)
        elif kind == 2:
            total += a * tree.total[1]
        elif kind == 3:
            out.append(total)
        else:
            raise ValueError('unknown query type')
    return out

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

tree=RangeAddSumTree([1,2,3,4]); rows=[('初始',[tree.query(i,i+1) for i in range(4)])]
for l,r,x in [(0,4,2),(1,3,-1),(2,4,5)]:
    tree.add(l,r,x); rows.append((f'[{l},{r})加{x}',[tree.query(i,i+1) for i in range(4)]))
show_table(['操作','实际逐点值'],rows)

操作,实际逐点值
初始,"[1, 2, 3, 4]"
"[0,4)加2","[3, 4, 5, 6]"
"[1,3)加-1","[3, 3, 4, 6]"
"[2,4)加5","[3, 3, 9, 11]"


## 7. 正确性、适用条件与复杂度

节点聚合值包含自身所有待下传更新，lazy仅表示孩子尚缺的部分，因此整段查询无需push。加法标记可相加，翻转标记按奇偶性合并。赋值后加法与加法后赋值不交换，不能把本章相加标记直接用于赋值拓展。

**代价：** 静态树建树O(n)时间空间；区间更新与查询O(log n)。查询1的输入区间闭合，转为内部半开区间；类型2、3只读根计数或总和为O(1)。这里实现静态节点；极大坐标域的动态开点与赋值标记属于拓展，不宣称已实现。

### 展开理解

为什么查询不用 push 也对：我们维护的约定是"节点的 total 永远是这个区间的真实和，lazy 只表示孩子还没收到的那部分更新"。整段覆盖的查询直接读 total，天然正确。什么时候这个约定会坏？只有当我们要去读孩子的时候——孩子的 total 可能还没收到父亲 lazy 里的账。所以 push 只出现在"部分相交、需要递归进孩子"之前，这正是代码里 push 仅有的两个位置。更新路径上，下去前 push 保证了旧账先结清，回来后 pull 保证了父亲重新等于孩子之和，于是约定在每个节点上都一直成立。

为什么加法标记可以叠加：给孩子补的账是"每个元素加 delta"，欠两笔就等价于欠一笔 delta1+delta2，加法对加法可合并。翻转标记为什么用异或：一段翻转两次等于原样，翻转奇数次等于翻转一次，异或恰好表达这种奇偶性；而且"先加后翻"和"先翻后加"在这两个类里不会混用（各自独立的树），互不干扰。

复杂度：建树每个节点访问一次，O(n) 时间和空间（数组开 4n）。区间更新和区间查询的递归每一层最多产生四个"部分相交"的子问题（左边界切一刀、右边界切一刀），所以访问的节点数是 O(log n) 量级。n = 10^5 时一次操作约访问一百多个节点、做几百次基本运算，十万次操作也就是千万级基本动作，秒级以内。类型 2、3 查询只读根或一个变量，是 O(1)。要提醒的是：这套是静态数组实现，坐标大到 10^9 时 4n 数组开不下，得用离线压缩或动态开点（拓展内容，本章未实现）。

## 8. 单元测试：每个用例在检查什么

- `flip=FlipCountTree([1,0,1]); flip.flip(1,2); assert flip.count(0,3)==3`：正常值测试。翻转区间 `[1,2)` 后数组变成 `[1,1,1]`，全区间数 1 得 3，验证翻转计数公式（长度减原 ones）。
- `flip.flip(0,3); flip.flip(0,3); assert flip.count(0,3)==3`：特殊值测试——同一个区间连翻两次，结果应完全不变。它专门打"标记按异或复合、两次抵消"这个点；如果 lazy 处理错了（比如翻两次变成加两次），这里立刻爆。
- `assert handle_sum_queries([1,0,1],[0,0,0],[[1,1,1],[2,1,0],[3,0,0]])==[3]`：整题流程测试。先翻转闭区间 `[1,1]`（注意题目给的是闭区间，代码转成半开 `[1,2)`），nums1 变 `[1,1,1]`；再执行类型 2：total += 1 × 3 = 3；类型 3 输出 3。它同时测了闭区间转半开和根节点 O(1) 读数。
- 中段随机对拍（种子 137）：n 从 1 到 19，同时维护加法树（值域 [-3,3]）和翻转树（0/1 串）。每轮先做一次随机区间加 + 随机区间翻转（用 for 循环同步更新暴力数组），再做一次随机区间查询，断言两棵树分别等于 `sum(a[x:y])` 和 `sum(bits[x:y])`。每轮 n×100 次交错更新查询，覆盖了"更新叠加更新""更新后立即查询""空区间（l==r）"等情形，比手写用例全面得多。
- `assert RangeAddSumTree([]).query(0,0)==0`：边界测试，空数组建树不崩、空查询返回 0（验证 `4*max(1,0)` 和 `if self.n:` 两处兜底）。

In [5]:
flip = FlipCountTree([1, 0, 1])

flip.flip(1, 2)

assert flip.count(0, 3) == 3

flip.flip(0, 3)

flip.flip(0, 3)

assert flip.count(0, 3) == 3

assert handle_sum_queries([1, 0, 1], [0, 0, 0], [[1, 1, 1], [2, 1, 0], [3, 0, 0]]) == [3]

from random import Random

rng = Random(137)

for n in range(1, 20):
    a = [rng.randrange(-3, 4) for _ in range(n)]
    bits = [rng.randrange(2) for _ in range(n)]
    st = RangeAddSumTree(a)
    ft = FlipCountTree(bits)
    for _ in range(100):
        l = rng.randrange(n + 1)
        r = rng.randrange(l, n + 1)
        delta = rng.randrange(-4, 5)
        st.add(l, r, delta)
        ft.flip(l, r)
        for i in range(l, r):
            a[i] += delta
            bits[i] ^= 1
        x = rng.randrange(n + 1)
        y = rng.randrange(x, n + 1)
        assert st.query(x, y) == sum(a[x:y]) and ft.count(x, y) == sum(bits[x:y])

assert RangeAddSumTree([]).query(0, 0) == 0

print('N137: 所有本章断言通过')

N137: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 加入赋值标记并说明与加法不可交换。

**练习 2：** 比较离线压缩和动态开点。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1：加入赋值标记并说明与加法不可交换。** 提示：赋值（整段改成某个值）也是懒标记，但它和加法不能像加法+加法那样直接合并。手算一个反例就明白：某元素先"加 2"再"赋 5"，最终是 5；先"赋 5"再"加 2"，最终是 7——顺序不同结果不同，所以节点上同时有赋值标记和加法标记时必须规定谁覆盖谁（惯例：赋值标记到来时清空加法标记）。实现时给节点加一个"当前赋的值 + 是否有赋值标记"的字段，重写 _apply 和 _push 里两个标记的复合逻辑。用 `add(0,2,2)`、`assign(0,2,5)`、`assign 后再 add` 三步小序列验证。

**练习 2：比较离线压缩和动态开点。** 提示：想象坐标范围是 `[0, 10^9]` 但只有 10^5 次操作，静态 4n 数组开不下。两条路：(a) 离线压缩——先把所有操作的端点收起来（N134 的坐标压缩），把大坐标映射成小秩再建静态树，代价是必须预先知道全部操作；(b) 动态开点——不预先建树，递归时"走到一个还不存在的节点才现场创建"，用字典或指针存孩子，可以在线处理但常数更大且实现更繁琐。建议各写一个只支持"区间加 + 全区间和查询"的玩具版本，用同一组大坐标操作对比结果，再讨论两者分别适用的场景。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
class RangeAddSumTree:

    def __init__(self, values):
        values = list(values)
        self.n = len(values)
        self.total = [0] * (4 * max(1, self.n))
        self.lazy = [0] * len(self.total)

        def build(node, l, r):
            if r - l == 1:
                self.total[node] = values[l]
                return
            mid = (l + r) // 2
            build(node * 2, l, mid)
            build(node * 2 + 1, mid, r)
            self.total[node] = self.total[node * 2] + self.total[node * 2 + 1]
        if self.n:
            build(1, 0, self.n)

    def _apply(self, node, l, r, delta):
        self.total[node] += (r - l) * delta
        self.lazy[node] += delta

    def _push(self, node, l, r):
        if self.lazy[node] and r - l > 1:
            mid = (l + r) // 2
            tag = self.lazy[node]
            self._apply(node * 2, l, mid, tag)
            self._apply(node * 2 + 1, mid, r, tag)
            self.lazy[node] = 0

    def add(self, left, right, delta):
        if not 0 <= left <= right <= self.n:
            raise IndexError('invalid half-open range')
        if left == right:
            return

        def update(node, l, r):
            if right <= l or r <= left:
                return
            if left <= l and r <= right:
                self._apply(node, l, r, delta)
                return
            self._push(node, l, r)
            mid = (l + r) // 2
            update(node * 2, l, mid)
            update(node * 2 + 1, mid, r)
            self.total[node] = self.total[node * 2] + self.total[node * 2 + 1]
        update(1, 0, self.n)

    def query(self, left, right):
        if not 0 <= left <= right <= self.n:
            raise IndexError('invalid half-open range')
        if left == right:
            return 0

        def get(node, l, r):
            if right <= l or r <= left:
                return 0
            if left <= l and r <= right:
                return self.total[node]
            self._push(node, l, r)
            mid = (l + r) // 2
            return get(node * 2, l, mid) + get(node * 2 + 1, mid, r)
        return get(1, 0, self.n)

class FlipCountTree(RangeAddSumTree):

    def __init__(self, values):
        values = list(values)
        if any((x not in (0, 1) for x in values)):
            raise ValueError('binary values required')
        super().__init__(values)

    def _apply(self, node, l, r, delta):
        if delta & 1:
            self.total[node] = r - l - self.total[node]
            self.lazy[node] ^= 1

    def flip(self, left, right):
        self.add(left, right, 1)

    def count(self, left, right):
        return self.query(left, right)

def handle_sum_queries(nums1, nums2, queries):
    tree = FlipCountTree(nums1)
    total = sum(nums2)
    out = []
    for kind, a, b in queries:
        if kind == 1:
            tree.flip(a, b + 1)
        elif kind == 2:
            total += a * tree.total[1]
        elif kind == 3:
            out.append(total)
        else:
            raise ValueError('unknown query type')
    return out

# 示例与回归测试
flip = FlipCountTree([1, 0, 1])

flip.flip(1, 2)

assert flip.count(0, 3) == 3

flip.flip(0, 3)

flip.flip(0, 3)

assert flip.count(0, 3) == 3

assert handle_sum_queries([1, 0, 1], [0, 0, 0], [[1, 1, 1], [2, 1, 0], [3, 0, 0]]) == [3]

from random import Random

rng = Random(137)

for n in range(1, 20):
    a = [rng.randrange(-3, 4) for _ in range(n)]
    bits = [rng.randrange(2) for _ in range(n)]
    st = RangeAddSumTree(a)
    ft = FlipCountTree(bits)
    for _ in range(100):
        l = rng.randrange(n + 1)
        r = rng.randrange(l, n + 1)
        delta = rng.randrange(-4, 5)
        st.add(l, r, delta)
        ft.flip(l, r)
        for i in range(l, r):
            a[i] += delta
            bits[i] ^= 1
        x = rng.randrange(n + 1)
        y = rng.randrange(x, n + 1)
        assert st.query(x, y) == sum(a[x:y]) and ft.count(x, y) == sum(bits[x:y])

assert RangeAddSumTree([]).query(0, 0) == 0

print('N137: 所有本章断言通过')

N137: 所有本章断言通过


## 11. 代表题与迁移

- **2569. Handling Sum Queries After Update（更新数组后处理求和查询）**：本章 canonical 题，`handle_sum_queries` 就是它的直接解：类型 1 练区间翻转（FlipCountTree 的 `_apply`），类型 2 练"区间统计 + 外部总和"的组合，类型 3 练 O(1) 读根。
- **715. Range Module（Range 模块）**：extension 题，练"区间标记的添加与撤销"（这里用翻转/赋值的思想跟踪区间是否被覆盖），动态开点或离线压缩正是它的经典出路，对应练习 2。
- **732. My Calendar III（我的日程安排 III）**：extension 题，把"区间加 1 / 区间减 1"的懒标记打在时间轴上，全局最大值就是最大同时预订数，练的是 range add 与聚合查询的迁移。

**题面入口：** [2569](https://leetcode.com/problems/handling-sum-queries-after-update/)

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。